# M8a — Non-circle synthetic shapes (reproduction)

Reproduces the headline tables from `docs/insights/m8a_non_circle_shapes.md`:
- pre-registered scoring (PASS / FAIL per criterion per model)
- PMR by (shape × object_level) — H1 ramp
- PMR by (shape × label_role) — H7
- GAR by (shape × label_role) — H7-GAR
- visual-saturation paired-delta

Stim dir: `inputs/m8a_qwen_20260425-091713_8af4836f/` (400 stim, single-shape baseline + 4 non-circle shapes).
Run dirs: 4 outputs (Qwen labeled / Qwen label-free / LLaVA labeled / LLaVA label-free).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from physical_mode.metrics.pmr import score_rows
from physical_mode.inference.prompts import LABELS_BY_SHAPE

OUT = PROJECT_ROOT / 'outputs'
Q_LBL = pd.read_json(OUT / 'm8a_qwen_20260425-092423_bf03832e' / 'predictions.jsonl', lines=True)
Q_NL  = pd.read_json(OUT / 'm8a_qwen_label_free_20260425-094239_26c66949' / 'predictions.jsonl', lines=True)
L_LBL = pd.read_json(OUT / 'm8a_llava_20260425-095133_a2b5f318' / 'predictions.jsonl', lines=True)
L_NL  = pd.read_json(OUT / 'm8a_llava_label_free_20260425-100253_99a20dd8' / 'predictions.jsonl', lines=True)

for name, df in [('Q_LBL', Q_LBL), ('Q_NL', Q_NL), ('L_LBL', L_LBL), ('L_NL', L_NL)]:
    print(f'{name}: {len(df)} rows')

Q_LBL = score_rows(Q_LBL); Q_NL = score_rows(Q_NL); L_LBL = score_rows(L_LBL); L_NL = score_rows(L_NL)

def role(shape, label):
    p, a, e = LABELS_BY_SHAPE[shape]
    return 'physical' if label == p else 'abstract' if label == a else 'exotic' if label == e else label

Q_LBL['label_role'] = [role(s, l) for s, l in zip(Q_LBL['shape'], Q_LBL['label'])]
L_LBL['label_role'] = [role(s, l) for s, l in zip(L_LBL['shape'], L_LBL['label'])]

## Headline 1 — H1 ramp (PMR by shape × object_level)

Pre-registered: ≥4/5 shapes with `PMR(textured) − PMR(line) ≥ 0.05` AND no internal inversion >0.05.

Result: Qwen 3/5 ✗, LLaVA 4/5 ✓.

In [ ]:
def pmr_ramp(df):
    return df.groupby(['shape', 'object_level'])['pmr'].mean().unstack()[['line','filled','shaded','textured']].round(3)

print('Qwen2.5-VL-7B'); print(pmr_ramp(Q_LBL))
print()
print('LLaVA-1.5-7B'); print(pmr_ramp(L_LBL))

## Headline 2 — H7 (PMR by shape × label_role)

Pre-registered: ≥3/5 shapes with `PMR(physical) − PMR(abstract) ≥ 0.05`.

Result: Qwen 1/5 ✗, LLaVA 4/5 ✓ (triangle is the LLaVA outlier — `wedge` is a weak physical label).

In [ ]:
def pmr_role(df):
    return df.groupby(['shape', 'label_role'])['pmr'].mean().unstack()[['physical','abstract','exotic']].round(3)

print('Qwen2.5-VL-7B'); print(pmr_role(Q_LBL))
print()
print('LLaVA-1.5-7B'); print(pmr_role(L_LBL))

## Headline 3 — H7-GAR (GAR by shape × label_role at bg ∈ {ground, scene})

Pre-registered: ≥3/5 shapes with `GAR(physical) ≥ GAR(abstract)`.

Result: Qwen 1/5 ✗, LLaVA 5/5 ✓.

In [ ]:
def gar_role(df):
    g = df[df['bg_level'].isin(['ground', 'scene'])]
    return g.groupby(['shape', 'label_role'])['gar'].mean().unstack()[['physical','abstract','exotic']].round(3)

print('Qwen2.5-VL-7B'); print(gar_role(Q_LBL))
print()
print('LLaVA-1.5-7B'); print(gar_role(L_LBL))

## Headline 4 — visual-saturation paired-delta (the M6 r2 prediction)

Pre-registered: Qwen ±0.05 on `physical` for ≥3/5 shapes; LLaVA ≥+0.05 on `physical` for ≥3/5 shapes.

Result: Qwen 3/5 ✓ borderline (square is a clear suppression at -0.20), LLaVA 5/5 ✓ (range +0.125 to +0.625).

Together with PMR(_nolabel) baseline (Qwen 0.78–0.93 vs LLaVA 0.075–0.288), this **is** the visual-saturation hypothesis: the saturated encoder leaves no behavioral headroom for labels to operate; the unsaturated encoder leaves all of it.

In [ ]:
shapes = ['circle','square','triangle','hexagon','polygon']
roles  = ['physical','abstract','exotic']

print('=== PMR(_nolabel) baseline ===')
print(pd.DataFrame({
    'Qwen':  Q_NL.groupby('shape')['pmr'].mean().round(3),
    'LLaVA': L_NL.groupby('shape')['pmr'].mean().round(3),
}))

def paired_delta(lbl, nl):
    rows = {}
    for s in shapes:
        s_nl = nl[nl['shape'] == s].groupby('sample_id')['pmr'].mean().rename('_nolabel')
        for r in roles:
            s_l = lbl[(lbl['shape'] == s) & (lbl['label_role'] == r)].groupby('sample_id')['pmr'].mean().rename(r)
            j = pd.concat([s_nl, s_l], axis=1).dropna()
            rows[(s, r)] = float((j[r] - j['_nolabel']).mean())
    return pd.DataFrame({s: {r: rows[(s, r)] for r in roles} for s in shapes}).T.round(3)

print('\n=== Qwen paired-deltas ===')
print(paired_delta(Q_LBL, Q_NL))
print('\n=== LLaVA paired-deltas ===')
print(paired_delta(L_LBL, L_NL))

## Bottom line

- Qwen (saturated encoder) fails 3 of 4 pre-registered criteria.
- LLaVA (unsaturated encoder) passes 4 of 4.
- The asymmetry is itself the cross-shape validation of the visual-saturation hypothesis from M6 r2 — predicted, not just consistent.